In [13]:
from netgen.occ import *
import ngsolve
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np
from tqdm import tqdm

In [14]:
pfeiler11=Box((0,0,0), (1,1,30))
pfeiler12=Box((11,0,0),(12,1,30))
strebe11=Box((1,0.2,15),(11,0.8,18))
strebe12=Box((1,0.2,26),(11,0.8,29))

pfeiler21=Box((0,30,0), (1,1+30,30))
pfeiler22=Box((11,30,0),(12,1+30,30))
strebe21=Box((1,0.2+30,15),(11,0.8+30,18))
strebe22=Box((1,0.2+30,26),(11,0.8+30,29))
road1=Box((1,1,10),(11,-5,11))
road2=Box((1,30,10),(11,35,11))

road=Box((1,-5,10),(11,35,11))

strebe11.faces.name="fix"
strebe12.faces.name="fix"
strebe21.faces.name="fix"
strebe22.faces.name="fix"

pfeiler11.faces.name="fix"
pfeiler12.faces.name="fix"
pfeiler21.faces.name="fix"
pfeiler22.faces.name="fix"

road.faces.Min(Y).name = "fix"
road.faces.Max(Y).name = "fix"
road.faces.Max(X).name = "force"
road.faces.Max(Z).name = "force"

geo=pfeiler11+pfeiler12+strebe11+strebe12
geo+=pfeiler21+pfeiler22+strebe21+strebe22
geo+=road

Draw(geo, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3,…

BaseWebGuiScene

In [15]:
from ngsolve import *
from ngsolve.webgui import Draw
mesh = Mesh(OCCGeometry(geo).GenerateMesh(maxh=1)).Curve(3)
Draw(mesh, height="3vh");

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

In [16]:
dt = 0.1
beta = .25 # 1/4 limit for unconditional stability, set higher for more damping
gamma = .5 # trapezoidal rule

E, nu = 210, 0.2
mu  = E / 2 / (1+nu)
lam = E * nu / ((1+nu)*(1-2*nu))

def Stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(3)

In [17]:
t = Parameter(0.0)
fes = VectorH1(mesh, order=3, dirichlet="fix")
u,v = fes.TnT()
gfu = GridFunction(fes)
gfv = GridFunction(fes)
gfa = GridFunction(fes)

with TaskManager():
    m_int = u*v*dx
    m = BilinearForm(m_int)
    m.Assemble()
    a_int = InnerProduct(Stress(Sym(Grad(u))), Sym(Grad(v))).Compile()*dx
    a = BilinearForm(a_int)
    a.Assemble()
    # amp1 = 1
    # amp2 = 1
    # c_int = amp1*u*v*dx + amp2*grad(u)*v*dx
    # c = BilinearForm(amp1* + amp2*)
    # c.Assemble()
    mstar = BilinearForm(fes)
    mstar += m_int
    #mstar += gamma * dt * c_int
    mstar += beta * dt**2 * a_int
    pre = MultiGridPreconditioner(mstar, "bddc") # Adds factor 8 Speedup for some reason ...
    mstar.Assemble()
    

In [18]:
omega = 0.7/2 # resoncance with  is around 1.5
force = CF( (1,0,-1))*cos(2*np.pi*omega*t) 
f = LinearForm(force*v*ds("force"))

In [19]:
from ngsolve.krylovspace import CGSolver

acc_inv = CGSolver(mstar.mat, pre, printrates=False, tol=1e-8)

$$ a^{n} = \left(\frac{\partial^2 u}{\partial t^2}\right)^{n} = \mathrm{gfa^{n}} $$
$$ v^{n} = \left(\frac{\partial u}{\partial t}\right)^{n} = \mathrm{gfv^{n}} $$

So what we do here ([Wikipedia](https://en.wikipedia.org/wiki/Newmark-beta_method)):



In [20]:
gfa.Set(CF((0,0,0)))
gfv.Set(CF((0,0,0)))
gfu.Set(CF((0,0,0)))

In [21]:
gfut = GridFunction(gfu.space,multidim=0)

fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
gfstress = GridFunction(fesstress)
gfstresst = GridFunction(fesstress, multidim=0)

tend = t.Get()+7
steps = int((tend - t.Get()) / dt)

with TaskManager():
    for _ in tqdm(range(steps), desc="Time-stepping", ncols=100, unit="step"):
        f.Assemble()

        rhs_acceleration = f.vec - a.mat*(gfu.vec + dt*gfv.vec + .5*dt**2*(1-2*beta)*gfa.vec) # - c.mat*(gfv.vec + (1-gamma)*gfa.vec) 
        
        new_acceleration = acc_inv * rhs_acceleration
        
        gfu.vec.data = gfu.vec + dt*gfv.vec + 0.5*dt**2*((1-2*beta)*gfa.vec + 2*beta*new_acceleration)
        
        gfv.vec.data = gfv.vec + dt*((1-gamma)*gfa.vec + gamma*new_acceleration)
        
        gfa.vec.data = new_acceleration
        
        gfut.AddMultiDimComponent(gfu.vec)
        
        gfstress.Interpolate(Stress(Sym(Grad(gfu))))
        
        gfstresst.AddMultiDimComponent(gfstress.vec)
        
        t.Set(t.Get()+dt)
    

Time-stepping: 100%|██████████████████████████████████████████████| 70/70 [01:20<00:00,  1.15s/step]


In [22]:
with TaskManager():
    fesstress = MatrixValued(H1(mesh,order=3), symmetric=True)
    gfstress = GridFunction(fesstress)
    gfstress.Interpolate (Stress(Sym(Grad(gfu))))

In [23]:
Draw (gfut, mesh, height="3vh", deformation=gfut, interpolate_multidim=True, animate=True)

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…

BaseWebGuiScene

In [24]:
Draw(gfstresst, mesh, height="3vh", deformation=gfut, interpolate_multidim=True, animate=True);

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.240…